# Neural IR Exercise dengan BERT Cross-Encoder
Dosen: Zico Pratama Putra

Kelompok:
- 14250028 - Hanief Fathul Bahri Ahmad
- 14250029 - Irfan Wibowo
- 14250027 - Muhammad Arief Nadhofa

Tanggal: ---

# **Setup All Import**

In [37]:
# Download data dan src dari GitHub (uncomment jika di Colab)
!rm -rf IR-BERT-Transformer data src
!git clone https://github.com/teranixbq/IR-BERT-Transformer.git
!mv IR-BERT-Transformer/data ./data
!mv IR-BERT-Transformer/src ./src
!rm -rf IR-BERT-Transformer
!pip install ir_datasets


Cloning into 'IR-BERT-Transformer'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 141 (delta 68), reused 66 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 232.99 KiB | 7.77 MiB/s, done.
Resolving deltas: 100% (68/68), done.


In [38]:
from src.judgement_aggregation import load_raw_judgements, aggregate_judgements, save_qrels, compute_annotator_reliability
from src.bert_cross_encoder import BERTCrossEncoder
from transformers import pipeline
import pandas as pd
import numpy as np
from tqdm import tqdm
import ir_datasets
import os
import random
os.makedirs("dataset_marco", exist_ok=True)

# **PART 1 : FiRA Judgement Aggregation**

## A. Setup

In [39]:
DEST = "/content/data/"
RAW_JUDGEMENTS = DEST + "fira_raw_judgements.tsv"
BASELINE_2022 = DEST + "fira-2022.baseline-qrels.tsv"

## B. Load Raw

In [40]:
df = load_raw_judgements(RAW_JUDGEMENTS)

Loaded 237 raw judgements
Columns: ['query_id', 'doc_id', 'judgement', 'confidence', 'annotator_id', 'duration_ms']
   query_id doc_id  judgement  confidence annotator_id  duration_ms
0         1   d1_1          3        0.85       User_5        30289
1         1   d1_1          3        0.94       User_0        83482
2         1   d1_1          3        0.92       User_4        16165
3         1   d1_2          2        0.81       User_0        38062
4         1   d1_2          1        0.92       User_5        12851


## C. Helper Function

In [41]:
def compare_with_baseline(
    aggregated_df: pd.DataFrame,
    baseline_qrels: pd.DataFrame
):
    """
    Membandingkan advanced aggregation dengan baseline qrels.

    Metrik:
    - matched query-doc pairs
    - exact match
    - exact match rate
    - mean absolute difference
    """

    merged = aggregated_df.merge(
        baseline_qrels[["query_id", "doc_id", "baseline_score"]],
        on=["query_id", "doc_id"],
        how="inner"
    )

    merged["score_diff"] = (
        merged["score"] -
        merged["baseline_score"]
    )

    merged["abs_diff"] = merged["score_diff"].abs()

    total_pairs = len(merged)
    exact_match = (merged["score_diff"] == 0).sum()

    exact_match_rate = (
        exact_match / total_pairs
        if total_pairs > 0
        else 0
    )

    mean_abs_diff = (
        merged["abs_diff"].mean()
        if total_pairs > 0
        else 0
    )

    print("=== Comparison with Baseline Qrels ===")
    print(f"Matched query-doc pairs  : {total_pairs}")
    print(f"Exact match              : {exact_match}")
    print(f"Exact match rate         : {exact_match_rate:.4f}")
    print(f"Mean absolute difference : {mean_abs_diff:.4f}")

    print("\nDistribusi perbedaan score:")
    print(merged["score_diff"].value_counts().sort_index())

    return merged

## 1.1 Baseline - Simple Majority Vote

In [42]:
agg_maj = aggregate_judgements(df, method='majority')
agg_maj.head(20)

Aggregated into 79 unique query-doc pairs


,query_id,doc_id,score,num_judgements,mean_score,median_score,std_score
0,1,d1_1,3,3,3.000000,3.0,0.000000
1,1,d1_2,2,3,1.666667,2.0,0.471405
2,1,d1_3,1,3,1.333333,1.0,0.471405
3,1,d1_4,0,3,0.000000,0.0,0.000000
4,1,d1_5,0,3,0.000000,0.0,0.000000
5,1,d1_6,0,3,0.000000,0.0,0.000000
6,1,d1_7,1,3,1.000000,1.0,0.000000
7,1,d1_8,1,3,0.666667,1.0,0.471405
8,2,d2_1,3,3,2.666667,3.0,0.471405
9,2,d2_2,2,3,2.333333,2.0,0.471405


## 1.2 Confidence-Based Weighting
Bobot berdasarkan seberapa yakin annotator

In [43]:
agg_weighted = aggregate_judgements(df, method='advanced', strategy='confidence_weighted')
agg_weighted.head(20)

Aggregated into 79 unique query-doc pairs


,query_id,doc_id,score,num_judgements,mean_score,median_score,std_score
0,1,d1_1,3,3,3.000000,3.0,0.000000
1,1,d1_2,2,3,1.666667,2.0,0.471405
2,1,d1_3,1,3,1.333333,1.0,0.471405
3,1,d1_4,0,3,0.000000,0.0,0.000000
4,1,d1_5,0,3,0.000000,0.0,0.000000
5,1,d1_6,0,3,0.000000,0.0,0.000000
6,1,d1_7,1,3,1.000000,1.0,0.000000
7,1,d1_8,1,3,0.666667,1.0,0.471405
8,2,d2_1,3,3,2.666667,3.0,0.471405
9,2,d2_2,2,3,2.333333,2.0,0.471405


## 1.3 Reliability-Weighted Voting
Bobot dari track record annotator — seberapa sering ia setuju dengan suara terbanyak (majority vote). Annotator yang sering benar dapat bobot lebih besar.

In [44]:
reliability = compute_annotator_reliability(df)
reliability

{'User_0': 0.7692307692307693,
 'User_1': 0.8372093023255814,
 'User_2': 0.7804878048780488,
 'User_3': 0.7659574468085106,
 'User_4': 0.8275862068965517,
 'User_5': 0.7631578947368421}

In [45]:
agg_reliability = aggregate_judgements(df, method='advanced', strategy='annotator_reliability', annotator_reliability=reliability)
agg_reliability.head(20)

Aggregated into 79 unique query-doc pairs


,query_id,doc_id,score,num_judgements,mean_score,median_score,std_score
0,1,d1_1,3,3,3.000000,3.0,0.000000
1,1,d1_2,2,3,1.666667,2.0,0.471405
2,1,d1_3,1,3,1.333333,1.0,0.471405
3,1,d1_4,0,3,0.000000,0.0,0.000000
4,1,d1_5,0,3,0.000000,0.0,0.000000
5,1,d1_6,0,3,0.000000,0.0,0.000000
6,1,d1_7,1,3,1.000000,1.0,0.000000
7,1,d1_8,1,3,0.666667,1.0,0.471405
8,2,d2_1,3,3,2.666667,3.0,0.471405
9,2,d2_2,2,3,2.333333,2.0,0.471405


## 1.4 Bandingkan Majority Dengan Advanced

### 1.4.1 Majority & Advanced (Confidence-Based Weighting)

In [46]:
comparison_df1 = agg_maj.merge(
    agg_weighted,
    on=["query_id", "doc_id"],
    suffixes=("_majority", "_advanced")
)

comparison_df1["score_diff"] = (
    comparison_df1["score_advanced"] -
    comparison_df1["score_majority"]
)

print("Jumlah query-doc pair:", len(comparison_df1))
print("Jumlah score berbeda :", (comparison_df1["score_diff"] != 0).sum())

print("\nDistribusi perbedaan score:")
print(comparison_df1["score_diff"].value_counts().sort_index())

display(comparison_df1.head())

Jumlah query-doc pair: 79
Jumlah score berbeda : 6

Distribusi perbedaan score:
score_diff
-1     4
 0    73
 1     2
Name: count, dtype: int64


,query_id,doc_id,score_majority,num_judgements_majority,mean_score_majority,median_score_majority,std_score_majority,score_advanced,num_judgements_advanced,mean_score_advanced,median_score_advanced,std_score_advanced,score_diff
0,1,d1_1,3,3,3.000000,3.0,0.000000,3,3,3.000000,3.0,0.000000,0
1,1,d1_2,2,3,1.666667,2.0,0.471405,2,3,1.666667,2.0,0.471405,0
2,1,d1_3,1,3,1.333333,1.0,0.471405,1,3,1.333333,1.0,0.471405,0
3,1,d1_4,0,3,0.000000,0.0,0.000000,0,3,0.000000,0.0,0.000000,0
4,1,d1_5,0,3,0.000000,0.0,0.000000,0,3,0.000000,0.0,0.000000,0


### 1.4.2 Majority & Advanced (Reliability-Weighted Voting)

In [47]:
comparison_df2 = agg_maj.merge(
    agg_reliability,
    on=["query_id", "doc_id"],
    suffixes=("_majority", "_advanced")
)

comparison_df2["score_diff"] = (
    comparison_df2["score_advanced"] -
    comparison_df2["score_majority"]
)

print("Jumlah query-doc pair:", len(comparison_df2))
print("Jumlah score berbeda :", (comparison_df2["score_diff"] != 0).sum())

print("\nDistribusi perbedaan score:")
print(comparison_df2["score_diff"].value_counts().sort_index())

display(comparison_df2.head())

Jumlah query-doc pair: 79
Jumlah score berbeda : 6

Distribusi perbedaan score:
score_diff
-1     4
 0    73
 1     2
Name: count, dtype: int64


,query_id,doc_id,score_majority,num_judgements_majority,mean_score_majority,median_score_majority,std_score_majority,score_advanced,num_judgements_advanced,mean_score_advanced,median_score_advanced,std_score_advanced,score_diff
0,1,d1_1,3,3,3.000000,3.0,0.000000,3,3,3.000000,3.0,0.000000,0
1,1,d1_2,2,3,1.666667,2.0,0.471405,2,3,1.666667,2.0,0.471405,0
2,1,d1_3,1,3,1.333333,1.0,0.471405,1,3,1.333333,1.0,0.471405,0
3,1,d1_4,0,3,0.000000,0.0,0.000000,0,3,0.000000,0.0,0.000000,0
4,1,d1_5,0,3,0.000000,0.0,0.000000,0,3,0.000000,0.0,0.000000,0


## 1.5 Bandingkan BASELINE dengan Advanced

In [48]:
baseline_df = pd.read_csv(BASELINE_2022, sep=r"\s+", names=["query_id", "unused", "doc_id", "baseline_score"])

### 1.5.1 BASELINE & Advanced (Confidence - Based Weighting)

In [49]:
if BASELINE_2022 is not None:
    baseline_comparison_df = compare_with_baseline(
        agg_weighted,
        baseline_df
    )

    display(baseline_comparison_df.head())
else:
    print("Baseline comparison dilewati karena baseline qrels tidak tersedia.")

=== Comparison with Baseline Qrels ===
Matched query-doc pairs  : 79
Exact match              : 73
Exact match rate         : 0.9241
Mean absolute difference : 0.0759

Distribusi perbedaan score:
score_diff
-1     4
 0    73
 1     2
Name: count, dtype: int64


,query_id,doc_id,score,num_judgements,mean_score,median_score,std_score,baseline_score,score_diff,abs_diff
0,1,d1_1,3,3,3.000000,3.0,0.000000,3,0,0
1,1,d1_2,2,3,1.666667,2.0,0.471405,2,0,0
2,1,d1_3,1,3,1.333333,1.0,0.471405,1,0,0
3,1,d1_4,0,3,0.000000,0.0,0.000000,0,0,0
4,1,d1_5,0,3,0.000000,0.0,0.000000,0,0,0


### 1.5.2 BASELINE & Advanced (Reliability-Weighted Voting)

In [50]:
if BASELINE_2022 is not None:
    baseline_comparison_df = compare_with_baseline(
        agg_reliability,
        baseline_df
    )

    display(baseline_comparison_df.head())
else:
    print("Baseline comparison dilewati karena baseline qrels tidak tersedia.")

=== Comparison with Baseline Qrels ===
Matched query-doc pairs  : 79
Exact match              : 73
Exact match rate         : 0.9241
Mean absolute difference : 0.0759

Distribusi perbedaan score:
score_diff
-1     4
 0    73
 1     2
Name: count, dtype: int64


,query_id,doc_id,score,num_judgements,mean_score,median_score,std_score,baseline_score,score_diff,abs_diff
0,1,d1_1,3,3,3.000000,3.0,0.000000,3,0,0
1,1,d1_2,2,3,1.666667,2.0,0.471405,2,0,0
2,1,d1_3,1,3,1.333333,1.0,0.471405,1,0,0
3,1,d1_4,0,3,0.000000,0.0,0.000000,0,0,0
4,1,d1_5,0,3,0.000000,0.0,0.000000,0,0,0


## 1.7 Analisis Manual Disagreement Tinggi

In [51]:
def inspect_high_disagreement_cases(
    raw_df: pd.DataFrame,
    aggregated_df: pd.DataFrame,
    top_n=5
):
    """
    Menampilkan contoh query-doc pair dengan disagreement tertinggi.

    Tujuan:
    - memeriksa variasi judgement antar annotator,
    - melihat apakah final score advanced aggregation masuk akal,
    - digunakan untuk analisis laporan.
    """

    high_disagreement = aggregated_df.sort_values(
        by="std_score",
        ascending=False
    ).head(top_n)

    for _, row in high_disagreement.iterrows():
        qid = row["query_id"]
        did = row["doc_id"]

        group = raw_df[
            (raw_df["query_id"] == qid) &
            (raw_df["doc_id"].astype(str) == str(did))
        ]

        print("=" * 80)
        print(f"Query ID             : {qid}")
        print(f"Doc ID               : {did}")
        print(f"Aggregated score     : {row['score']}")
        print(f"Number of judgements : {row['num_judgements']}")
        print(f"Std score            : {row['std_score']:.2f}")
        print("\nRaw judgements:")
        display(group)

In [52]:
inspect_high_disagreement_cases(
    raw_df=df,
    aggregated_df=agg_weighted,
    top_n=5
)

Query ID             : 9
Doc ID               : d9_3
Aggregated score     : 1
Number of judgements : 3
Std score            : 0.94

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
198,9,d9_3,0,0.67,User_2,43300
199,9,d9_3,0,0.96,User_5,17720
200,9,d9_3,2,0.64,User_3,61968


Query ID             : 6
Doc ID               : d6_7
Aggregated score     : 1
Number of judgements : 3
Std score            : 0.94

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
138,6,d6_7,0,0.97,User_1,89656
139,6,d6_7,2,0.85,User_4,23359
140,6,d6_7,2,0.55,User_2,72910


Query ID             : 4
Doc ID               : d4_7
Aggregated score     : 1
Number of judgements : 3
Std score            : 0.94

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
90,4,d4_7,0,0.93,User_5,55279
91,4,d4_7,2,0.82,User_1,51363
92,4,d4_7,2,0.87,User_2,50752


Query ID             : 9
Doc ID               : d9_2
Aggregated score     : 2
Number of judgements : 3
Std score            : 0.94

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
195,9,d9_2,3,0.67,User_5,56548
196,9,d9_2,3,0.56,User_1,51132
197,9,d9_2,1,0.59,User_0,75483


Query ID             : 6
Doc ID               : d6_2
Aggregated score     : 2
Number of judgements : 3
Std score            : 0.82

Raw judgements:


,query_id,doc_id,judgement,confidence,annotator_id,duration_ms
123,6,d6_2,1,0.95,User_3,61692
124,6,d6_2,3,0.67,User_2,45065
125,6,d6_2,2,0.95,User_1,18826


## 1.6 Simpan Semua

In [53]:
save_qrels(agg_maj, DEST + "fira_aggregated.qrels")
save_qrels(agg_weighted, DEST + "fira_aggregated_confidence_weighted.qrels")
save_qrels(agg_reliability, DEST + "fira_aggregated_annotator_reliability.qrels")

Qrels saved to /content/data/fira_aggregated.qrels
Qrels saved to /content/data/fira_aggregated_confidence_weighted.qrels
Qrels saved to /content/data/fira_aggregated_annotator_reliability.qrels


# **Part 2: BERT Cross-Encoder Re-Ranking**

## A Setup Dataframe

In [54]:
df_tuples = pd.read_csv(DEST + "fira-2022.tuples.tsv", sep="\t",
                        names=["query_id", "doc_id", "query", "passage"])

# Skip kolom ke-2 (0/col), ambil qid, doc, score saja
qrels = pd.read_csv(DEST + "fira_aggregated.qrels", sep=r"\s+",
                    names=["query_id", "doc_id", "score"], usecols=[0, 2, 3])

df_merged_FiRA = df_tuples.merge(qrels, left_on=["query_id", "doc_id"], right_on=["query_id", "doc_id"])


df_merged_FiRA["label"] = (df_merged_FiRA["score"] >= 2).astype(int)

print(f"Total pairs: {len(df_merged_FiRA)}")
print(f"Relevant: {df_merged_FiRA['label'].sum()}, Irrelevant: {(1-df_merged_FiRA['label']).sum()}")
df_merged_FiRA.head()


Total pairs: 79
Relevant: 21, Irrelevant: 58


,query_id,doc_id,query,passage,score,label
0,1,d1_1,how to make a good cappuccino,"To make a perfect cappuccino, you need equal p...",3,1
1,1,d1_2,how to make a good cappuccino,Making a cappuccino at home requires an espres...,2,1
2,1,d1_3,how to make a good cappuccino,Cappuccino is an espresso-based coffee drink t...,1,0
3,1,d1_4,how to make a good cappuccino,Coffee beans are roasted to develop their flav...,0,0
4,1,d1_5,how to make a good cappuccino,The history of tea dates back to ancient China...,0,0


## B. Helper Function

### B.1 Evaluasi

In [55]:
def evaluate_reranker(reranker, candidate_df, qrels_df, k=10, batch_size=8, max_queries=None):
    """
    Evaluasi re-ranker pada candidate passages.

    candidate_df wajib punya:
    - query_id
    - doc_id
    - query
    - passage

    qrels_df wajib punya:
    - query_id
    - doc_id
    - score

    Returns: (results_dict, topk_df)
    """

    query_ids = candidate_df["query_id"].unique().tolist()

    if max_queries is not None:
        query_ids = query_ids[:max_queries]

    qrels_dict = {
        (row["query_id"], row["doc_id"]): int(row["score"])
        for _, row in qrels_df.iterrows()
    }

    all_mrr = []
    all_ndcg = []
    all_precision = []
    all_topk = []

    for qid in tqdm(query_ids, desc="Evaluating queries"):

        query_docs = candidate_df[
            candidate_df["query_id"] == qid
        ].copy()

        if len(query_docs) == 0:
            continue

        query_text = query_docs.iloc[0]["query"]

        passages = query_docs["passage"].tolist()
        doc_ids = query_docs["doc_id"].tolist()

        ranked_indices, scores = reranker.re_rank(
            query=query_text,
            passages=passages,
            batch_size=batch_size,
            verbose=False
        )

        ranked_doc_ids = [
            doc_ids[idx]
            for idx in ranked_indices
        ]

        ranked_relevances = [
            qrels_dict.get((qid, doc_id), 0)
            for doc_id in ranked_doc_ids
        ]

        all_mrr.append(mrr_at_k(ranked_relevances, k))
        all_ndcg.append(ndcg_at_k(ranked_relevances, k))
        all_precision.append(precision_at_k(ranked_relevances, k))

        for rank, idx in enumerate(ranked_indices[:k], 1):
            all_topk.append({
                "query_id": qid,
                "query": query_text,
                "doc_id": doc_ids[idx],
                "passage": passages[idx],
                "relevance_score": scores[idx],
                "rank": rank
            })

    results = {
        f"MRR@{k}": float(np.mean(all_mrr)) if all_mrr else 0.0,
        f"NDCG@{k}": float(np.mean(all_ndcg)) if all_ndcg else 0.0,
        f"Precision@{k}": float(np.mean(all_precision)) if all_precision else 0.0,
        "num_queries": len(all_mrr)
    }

    return results, pd.DataFrame(all_topk)



def evaluate_existing_order(
    candidate_df,
    qrels_df,
    k=10
):
    """
    Evaluasi urutan asli candidate passages sebagai baseline first-stage ranker.
    """

    query_ids = candidate_df["query_id"].unique().tolist()

    qrels_dict = {
        (row["query_id"], row["doc_id"]): int(row["score"])
        for _, row in qrels_df.iterrows()
    }

    all_mrr = []
    all_ndcg = []
    all_precision = []

    for qid in query_ids:

        query_docs = candidate_df[
            candidate_df["query_id"] == qid
        ].copy()

        doc_ids = query_docs["doc_id"].tolist()

        relevances = [
            qrels_dict.get((qid, doc_id), 0)
            for doc_id in doc_ids
        ]

        all_mrr.append(mrr_at_k(relevances, k))
        all_ndcg.append(ndcg_at_k(relevances, k))
        all_precision.append(precision_at_k(relevances, k))

    return {
        f"MRR@{k}": float(np.mean(all_mrr)) if all_mrr else 0.0,
        f"NDCG@{k}": float(np.mean(all_ndcg)) if all_ndcg else 0.0,
        f"Precision@{k}": float(np.mean(all_precision)) if all_precision else 0.0,
        "num_queries": len(all_mrr)
    }

### B.2 Metriks MRR@10, NDCG@10, Precision@10

In [56]:

def mrr_at_k(relevances, k=10):
    """
    MRR@K untuk satu query.
    """

    relevances = relevances[:k]

    for idx, rel in enumerate(relevances):
        if rel > 0:
            return 1.0 / (idx + 1)

    return 0.0


def dcg_at_k(relevances, k=10):
    """
    DCG@K.
    """

    relevances = np.asarray(relevances[:k])

    if len(relevances) == 0:
        return 0.0

    gains = (2 ** relevances - 1)
    discounts = np.log2(np.arange(len(relevances)) + 2)

    return np.sum(gains / discounts)


def ndcg_at_k(relevances, k=10):
    """
    NDCG@K.
    """

    dcg = dcg_at_k(relevances, k)
    ideal = dcg_at_k(sorted(relevances, reverse=True), k)

    if ideal == 0:
        return 0.0

    return dcg / ideal


def precision_at_k(relevances, k=10):
    """
    Precision@K.
    """

    relevances = relevances[:k]

    if len(relevances) == 0:
        return 0.0

    binary_rels = [
        1 if rel > 0 else 0
        for rel in relevances
    ]

    return sum(binary_rels) / k

## 2.1 FiRA Fine-tune BERT Cross-Encoder

In [58]:
reranker = BERTCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

reranker.train(
    df_merged_FiRA[["query", "passage", "label"]],
    epochs=5)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Model cross-encoder/ms-marco-MiniLM-L-6-v2 loaded on cuda
Epoch 1/5 — avg loss: 0.6521
Epoch 2/5 — avg loss: 0.4738
Epoch 3/5 — avg loss: 0.2976
Epoch 4/5 — avg loss: 0.3251
Epoch 5/5 — avg loss: 0.1555


## 2.3 FiRA - Re-rank dengan BERT Fine-Tuned

In [59]:
results_fira_BERT = []
for (qid, q), group in df_merged_FiRA.groupby(["query_id", "query"]):
    passages = group["passage"].tolist()
    doc_ids = group["doc_id"].tolist()
    labels = group["label"].tolist()

    ranked_idx, scores = reranker.re_rank(q, passages)

    ranked_docs = [doc_ids[i] for i in ranked_idx]
    ranked_scores = [scores[i] for i in ranked_idx]
    ranked_labels = [labels[i] for i in ranked_idx]

    results_fira_BERT.append({
        "query_id": qid,
        "query": q,
        "ranked_docs": ranked_docs,
        "scores": ranked_scores,
        "labels": ranked_labels,
    })

    print(f"Query {qid}")
    for rank, (doc, sc, lb) in enumerate(zip(ranked_docs, ranked_scores, ranked_labels), 1):
        print(f"  {rank}. {doc} (score={sc:.4f}, relevant={lb})")
    print()

Re-ranking: 100%|██████████| 1/1 [00:00<00:00, 12.22it/s]


Query 1
  1. d1_1 (score=0.9999, relevant=1)
  2. d1_2 (score=0.9994, relevant=1)
  3. d1_3 (score=0.9757, relevant=0)
  4. d1_7 (score=0.0291, relevant=0)
  5. d1_6 (score=0.0002, relevant=0)
  6. d1_8 (score=0.0001, relevant=0)
  7. d1_5 (score=0.0000, relevant=0)
  8. d1_4 (score=0.0000, relevant=0)



Re-ranking: 100%|██████████| 1/1 [00:00<00:00, 13.68it/s]


Query 2
  1. d2_1 (score=0.9996, relevant=1)
  2. d2_2 (score=0.9924, relevant=1)
  3. d2_5 (score=0.9853, relevant=0)
  4. d2_3 (score=0.7452, relevant=0)
  5. d2_4 (score=0.0738, relevant=0)
  6. d2_8 (score=0.0001, relevant=0)
  7. d2_7 (score=0.0000, relevant=0)
  8. d2_6 (score=0.0000, relevant=0)



Re-ranking: 100%|██████████| 1/1 [00:00<00:00, 13.59it/s]


...

Query 8
  1. d8_1 (score=0.9997, relevant=1)
  2. d8_7 (score=0.9992, relevant=0)
  3. d8_2 (score=0.9947, relevant=1)
  4. d8_5 (score=0.3787, relevant=0)
  5. d8_4 (score=0.0919, relevant=0)
  6. d8_3 (score=0.0062, relevant=0)
  7. d8_6 (score=0.0001, relevant=0)
  8. d8_8 (score=0.0000, relevant=0)



Re-ranking: 100%|██████████| 1/1 [00:00<00:00, 11.38it/s]


Query 9
  1. d9_1 (score=0.9999, relevant=1)
  2. d9_2 (score=0.9993, relevant=1)
  3. d9_7 (score=0.9950, relevant=0)
  4. d9_4 (score=0.7724, relevant=0)
  5. d9_3 (score=0.3507, relevant=0)
  6. d9_5 (score=0.0989, relevant=0)
  7. d9_6 (score=0.0178, relevant=0)
  8. d9_8 (score=0.0127, relevant=0)



Re-ranking: 100%|██████████| 1/1 [00:00<00:00, 14.51it/s]

Query 10
  1. d10_1 (score=0.9999, relevant=1)
  2. d10_2 (score=0.9994, relevant=1)
  3. d10_7 (score=0.9981, relevant=1)
  4. d10_3 (score=0.0607, relevant=0)
  5. d10_4 (score=0.0009, relevant=0)
  6. d10_8 (score=0.0003, relevant=0)
  7. d10_5 (score=0.0001, relevant=0)



## 2.3 Evaluasi ReRanked & First Stage Dengan Data FiRA

In [63]:
fira_baseline_results= evaluate_existing_order(
    candidate_df=df_tuples,
    qrels_df=qrels,
    k=10
)

fira_reranker_results ,_= evaluate_reranker(
    reranker=reranker,
    candidate_df=df_tuples,
    qrels_df=qrels,
    k=10,
    batch_size=8,
    max_queries=None
)

fira_comparison_df = pd.DataFrame([
    {
        "Dataset": "FiRA",
        "Model": "First-stage / Original Order",
        "MRR@10": fira_baseline_results["MRR@10"],
        "NDCG@10": fira_baseline_results["NDCG@10"],
        "Precision@10": fira_baseline_results["Precision@10"],
        "Num Queries": fira_baseline_results["num_queries"]
    },
    {
        "Dataset": "FiRA",
        "Model": "BERT Cross-Encoder Re-Ranker",
        "MRR@10": fira_reranker_results["MRR@10"],
        "NDCG@10": fira_reranker_results["NDCG@10"],
        "Precision@10": fira_reranker_results["Precision@10"],
        "Num Queries": fira_reranker_results["num_queries"]
    }
])

display(fira_comparison_df)

Evaluating queries: 100%|██████████| 10/10 [00:00<00:00, 12.68it/s]


,Dataset,Model,MRR@10,NDCG@10,Precision@10,Num Queries
0,FiRA,First-stage / Original Order,1.0,0.927538,0.45,10
1,FiRA,BERT Cross-Encoder Re-Ranker,1.0,0.937349,0.45,10


## 2.4 Evaluasi MS Marco
Hanya menggunakan 1000 sample query

### 2.4.1 Download MS Marco Top 1000

In [ ]:
!wget -P data/ https://msmarco.z22.web.core.windows.net/msmarcoranking/top1000.dev.tar.gz

--2026-06-17 14:26:57--  https://msmarco.z22.web.core.windows.net/msmarcoranking/top1000.dev.tar.gz
Resolving msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)... 57.150.146.100
Connecting to msmarco.z22.web.core.windows.net (msmarco.z22.web.core.windows.net)|57.150.146.100|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 687414398 (656M) [application/x-gzip]
Saving to: ‘data/top1000.dev.tar.gz’

top1000.dev.tar.gz  100%[===================>] 655.57M  4.74MB/s    in 2m 24s  

2026-06-17 14:29:22 (4.54 MB/s) - ‘data/top1000.dev.tar.gz’ saved [687414398/687414398]



In [ ]:
!tar -xzf data/top1000.dev.tar.gz -C /content/dataset_marco/

In [64]:
top1000_df = pd.read_csv(
    "/content/dataset_marco/top1000.dev",
    sep="\t", header=None,
    names=["qid", "pid", "query", "passage"]
)
top1000_df["qid"] = top1000_df["qid"].astype(str)
top1000_df["pid"] = top1000_df["pid"].astype(str)

print(f"Loaded {len(top1000_df)} rows")

Loaded 6668967 rows


### 2.4.2 Download ms marco-passage/dev/small

In [65]:
dataset = ir_datasets.load("msmarco-passage/dev/small")

qrels_list = []
for qid, docid, rel, _ in dataset.qrels_iter():
    qrels_list.append({"query_id": qid, "doc_id": docid, "score": rel})
qrels_df_msmarco = pd.DataFrame(qrels_list)

queries_dict = {qid: text for qid, text in dataset.queries_iter()}
print(f"Queries: {len(queries_dict)}, Qrels: {len(qrels_df_msmarco)}")

[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] If you have a local copy of https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/31644046b18952c1386cd4564ba2ae69
[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz
[INFO] [finished] https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: [04:52] [1.06GB] [3.61MB/s]


Queries: 6980, Qrels: 7437


### 2.4.3 Sample 1000 MS MARCO Queries + Build Candidates

In [75]:
qids_with_qrels = qrels_df_msmarco["query_id"].unique()
qids_in_top1000 = np.intersect1d(qids_with_qrels, top1000_df["qid"].unique())
qids_sample = qids_in_top1000[:1000]

candidate_df_msmarco = (
    top1000_df[top1000_df["qid"].isin(qids_sample)]
    .groupby("qid").head(100)
    .copy()
)
candidate_df_msmarco.columns = ["query_id", "doc_id", "query", "passage"]

qrels_sample = qrels_df_msmarco[qrels_df_msmarco["query_id"].isin(qids_sample)]

print(f"Queries: {len(qids_sample)}, Candidates: {len(candidate_df_msmarco)}, Qrels: {len(qrels_sample)}")

Queries: 1000, Candidates: 96946, Qrels: 1045


### 2.4.4 Evaluasi Reranker dengan MS Marco 1000 query

In [77]:
ms_results_ft, topk_df_ft = evaluate_reranker(
    reranker=reranker,
    candidate_df=candidate_df_msmarco,
    qrels_df=qrels_sample,
    k=10, batch_size=32
)
print("\nMS MARCO — BERT Fine-tuned (FiRA):", ms_results_ft)

Evaluating queries: 100%|██████████| 1000/1000 [12:24<00:00,  1.34it/s]


MS MARCO — BERT Fine-tuned (FiRA): {'MRR@10': 0.08199126984126984, 'NDCG@10': 0.0896684132779915, 'Precision@10': 0.011399999999999999, 'num_queries': 1000}


## 2.5 Evaluasi ALL

In [81]:
comparison_ms = pd.DataFrame([
    {
        "Dataset": "MS MARCO",
        "Model": "BERT Fine-tuned (FiRA)",
        "MRR@10": ms_results_ft["MRR@10"],
        "NDCG@10": ms_results_ft["NDCG@10"],
        "Precision@10": ms_results_ft["Precision@10"],
        "Num Queries": ms_results_ft["num_queries"]
    }
])

# Gabung dengan tabel FiRA (pastikan fira_comparison_df sudah ada)
final_df = pd.concat([fira_comparison_df, comparison_ms], ignore_index=True)
display(final_df)

,Dataset,Model,MRR@10,NDCG@10,Precision@10,Num Queries
0,FiRA,First-stage / Original Order,1.000000,0.927538,0.4500,10
1,FiRA,BERT Cross-Encoder Re-Ranker,1.000000,0.937349,0.4500,10
2,MS MARCO,BERT Fine-tuned (FiRA),0.081991,0.089668,0.0114,1000


# **Part 3: Extractive QA**

Pada bagian ini, BERT Cross-Encoder dari Part 2 dipakai untuk memilih passage paling relevan, lalu model Extractive QA mengambil potongan jawaban langsung dari passage tersebut.

Alur kerja:
1. Ambil candidate passage dari FiRA atau MS MARCO.
2. Re-rank candidate passage dengan BERT Cross-Encoder.
3. Jalankan Extractive QA pada top passage.
4. Evaluasi ranking dengan **MRR@10**, **NDCG@10**, dan **Precision@10**.
5. Evaluasi apakah passage sumber jawaban termasuk dokumen relevan berdasarkan qrels.

## 3.1 Konfigurasi Extractive QA

Silakan ubah `PART3_DATASET` menjadi `"MSMARCO"` jika ingin menggunakan candidate MS MARCO dari Part 2. Untuk laptop biasa, mulai dari `PART3_MAX_QUERIES = 5` atau `10` dahulu agar proses tidak terlalu berat.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

QA_MODEL_NAME = "deepset/roberta-base-squad2"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("QA model:", QA_MODEL_NAME)


class ManualExtractiveQAPipeline:
    def __init__(
        self,
        model_name: str,
        device: torch.device,
        max_seq_len: int = 384,
        doc_stride: int = 128,
        n_best_size: int = 20,
    ):
        self.device = device
        self.max_seq_len = max_seq_len
        self.doc_stride = doc_stride
        self.n_best_size = n_best_size

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def __call__(
        self,
        question: str,
        context: str,
        handle_impossible_answer: bool = True,
        max_answer_len: int = 50,
        **kwargs,
    ):
        question = str(question).strip()
        context = str(context).strip()

        if not question or not context:
            return {"answer": "", "score": 0.0, "start": None, "end": None}

        encoded = self.tokenizer(
            question,
            context,
            max_length=self.max_seq_len,
            truncation="only_second",
            stride=self.doc_stride,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding="max_length",
            return_tensors="pt",
        )

        offset_mapping = encoded.pop("offset_mapping")
        encoded.pop("overflow_to_sample_mapping", None)

        sequence_ids_per_feature = [
            encoded.sequence_ids(i) for i in range(encoded["input_ids"].shape[0])
        ]

        model_inputs = encoded.to(self.device)

        with torch.no_grad():
            outputs = self.model(**model_inputs)

        start_logits = outputs.start_logits.detach().cpu()
        end_logits = outputs.end_logits.detach().cpu()
        offset_mapping = offset_mapping.cpu().tolist()

        best_answer = {"answer": "", "score": 0.0, "start": None, "end": None}

        for feature_idx in range(start_logits.shape[0]):
            start_probs = torch.softmax(start_logits[feature_idx], dim=-1)
            end_probs = torch.softmax(end_logits[feature_idx], dim=-1)

            start_top = torch.topk(
                start_probs,
                k=min(self.n_best_size, start_probs.shape[0])
            ).indices.tolist()

            end_top = torch.topk(
                end_probs,
                k=min(self.n_best_size, end_probs.shape[0])
            ).indices.tolist()

            sequence_ids = sequence_ids_per_feature[feature_idx]
            offsets = offset_mapping[feature_idx]

            for start_idx in start_top:
                for end_idx in end_top:
                    if sequence_ids[start_idx] != 1 or sequence_ids[end_idx] != 1:
                        continue

                    if end_idx < start_idx:
                        continue

                    if (end_idx - start_idx + 1) > max_answer_len:
                        continue

                    char_start, _ = offsets[start_idx]
                    _, char_end = offsets[end_idx]

                    if char_end <= char_start:
                        continue

                    answer_text = context[char_start:char_end].strip()
                    if not answer_text:
                        continue

                    score = float(start_probs[start_idx] * end_probs[end_idx])

                    if score > best_answer["score"]:
                        best_answer = {
                            "answer": answer_text,
                            "score": score,
                            "start": int(char_start),
                            "end": int(char_end),
                        }

        return best_answer


qa_pipeline = ManualExtractiveQAPipeline(
    model_name=QA_MODEL_NAME,
    device=DEVICE,
    max_seq_len=384,
    doc_stride=128,
    n_best_size=20,
)

print("Manual Extractive QA siap digunakan.")

## 3.2 Siapkan Candidate Passage dan Qrels

Cell ini mengambil data yang sudah dibuat di Part 2. Format kolom akan diseragamkan menjadi:
- `query_id`
- `doc_id`
- `query`
- `passage`

Qrels digunakan sebagai label relevansi untuk evaluasi ranking dan sumber jawaban.

In [ ]:
import pandas as pd

# Memulihkan data yang hilang dari Part 2 agar Part 3 bisa berjalan
print("Memulihkan data dari file...")
try:
    # Load df_tuples dan qrels dari disk
    df_tuples = pd.read_csv("/content/data/fira-2022.tuples.tsv", sep="\t", names=["query_id", "doc_id", "query", "passage"])
    qrels = pd.read_csv("/content/data/fira_aggregated.qrels", sep=r"\s+", names=["query_id", "doc_id", "score"], usecols=[0, 2, 3])

    # Inisialisasi konfigurasi jika hilang
    if 'PART3_DATASET' not in globals(): PART3_DATASET = "FiRA"
    if 'PART3_MAX_QUERIES' not in globals(): PART3_MAX_QUERIES = 10
    if 'PART3_RERANK_TOP_K' not in globals(): PART3_RERANK_TOP_K = 10
    if 'PART3_QA_CONTEXT_TOP_K' not in globals(): PART3_QA_CONTEXT_TOP_K = 5
    if 'PART3_BATCH_SIZE' not in globals(): PART3_BATCH_SIZE = 8
    if 'QA_MIN_SCORE' not in globals(): QA_MIN_SCORE = 0.05

    print("✅ Data berhasil dimuat ulang.")

    # Menjalankan ulang persiapan data Part 3.2
    part3_candidate_df, part3_qrels_df = prepare_part3_data(
        dataset_name=PART3_DATASET,
        max_queries=PART3_MAX_QUERIES
    )

    if part3_candidate_df is not None:
        display(part3_candidate_df.head(2))
        print("Silakan jalankan cell re-ranking (3.4) dan QA (3.5) sekarang.")

except Exception as e:
    print(f"Gagal memuat data: {e}")

In [ ]:
def standardize_candidate_df(candidate_df: pd.DataFrame) -> pd.DataFrame:
    """Menyeragamkan nama kolom candidate passage."""
    df = candidate_df.copy()

    rename_map = {}
    if "qid" in df.columns:
        rename_map["qid"] = "query_id"
    if "pid" in df.columns:
        rename_map["pid"] = "doc_id"

    df = df.rename(columns=rename_map)

    required_cols = ["query_id", "doc_id", "query", "passage"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Kolom candidate_df belum lengkap: {missing_cols}")

    df = df[required_cols].copy()
    df["query_id"] = df["query_id"].astype(str)
    df["doc_id"] = df["doc_id"].astype(str)
    df["query"] = df["query"].fillna("").astype(str)
    df["passage"] = df["passage"].fillna("").astype(str)

    # Buang passage kosong agar QA tidak error.
    df = df[df["passage"].str.strip() != ""].reset_index(drop=True)
    return df


def standardize_qrels_df(qrels_df: pd.DataFrame) -> pd.DataFrame:
    """Menyeragamkan qrels menjadi query_id, doc_id, score."""
    df = qrels_df.copy()

    required_cols = ["query_id", "doc_id", "score"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Kolom qrels_df belum lengkap: {missing_cols}")

    df = df[required_cols].copy()
    df["query_id"] = df["query_id"].astype(str)
    df["doc_id"] = df["doc_id"].astype(str)
    df["score"] = pd.to_numeric(df["score"], errors="coerce").fillna(0).astype(int)
    return df


def prepare_part3_data(dataset_name="FiRA", max_queries=10):
    """
    Mengambil candidate dan qrels untuk Part 3.
    """
    dataset_key = dataset_name.upper().replace(" ", "")

    if dataset_key == "FIRA":
        # Cek apakah variabel sudah didefinisikan di global scope
        if "df_tuples" not in globals() or "qrels" not in globals():
            print("Error: Variabel 'df_tuples' atau 'qrels' tidak ditemukan.")
            print("Pastikan Anda telah menjalankan cell di Part 2 (Load Dataframe) sampai selesai.")
            return None, None
        candidate = standardize_candidate_df(df_tuples)
        qrels_eval = standardize_qrels_df(qrels)

    elif dataset_key in ["MSMARCO", "MSMARCO-PASSAGE"]:
        if "candidate_df_msmarco" not in globals() or "qrels_sample" not in globals():
            print("Error: Variabel 'candidate_df_msmarco' atau 'qrels_sample' tidak ditemukan.")
            return None, None
        candidate = standardize_candidate_df(candidate_df_msmarco)
        qrels_eval = standardize_qrels_df(qrels_sample)

    else:
        raise ValueError("dataset_name harus 'FiRA' atau 'MSMARCO'.")

    qrels_dict = {
        (row.query_id, row.doc_id): int(row.score)
        for row in qrels_eval.itertuples(index=False)
    }

    eligible_qids = []
    fallback_qids = []

    for qid, group in candidate.groupby("query_id", sort=False):
        fallback_qids.append(qid)
        doc_ids = group["doc_id"].tolist()
        has_relevant_doc = any(qrels_dict.get((qid, doc_id), 0) > 0 for doc_id in doc_ids)
        if has_relevant_doc:
            eligible_qids.append(qid)

    selected_qids = eligible_qids[:max_queries] if eligible_qids else fallback_qids[:max_queries]
    candidate = candidate[candidate["query_id"].isin(selected_qids)].copy().reset_index(drop=True)
    qrels_eval = qrels_eval[qrels_eval["query_id"].isin(selected_qids)].copy().reset_index(drop=True)

    print("Dataset              :", dataset_name)
    print("Selected queries      :", candidate["query_id"].nunique())
    return candidate, qrels_eval


part3_candidate_df, part3_qrels_df = prepare_part3_data(
    dataset_name=PART3_DATASET,
    max_queries=PART3_MAX_QUERIES
)

if part3_candidate_df is not None:
    display(part3_candidate_df.head())
    display(part3_qrels_df.head())

## 3.3 Fungsi Metrik Ranking

Metrik ini mengevaluasi urutan passage hasil re-ranking:
- **MRR@10**: posisi dokumen relevan pertama.
- **NDCG@10**: kualitas ranking berbasis graded relevance.
- **Precision@10**: proporsi dokumen relevan di top 10.

In [ ]:
def mrr_at_k(relevances, k=10):
    relevances = list(relevances)[:k]
    for idx, rel in enumerate(relevances):
        if rel > 0:
            return 1.0 / (idx + 1)
    return 0.0


def dcg_at_k(relevances, k=10):
    relevances = np.asarray(list(relevances)[:k])
    if len(relevances) == 0:
        return 0.0

    gains = (2 ** relevances - 1)
    discounts = np.log2(np.arange(len(relevances)) + 2)
    return float(np.sum(gains / discounts))


def ndcg_at_k(relevances, k=10):
    relevances = list(relevances)[:k]
    dcg = dcg_at_k(relevances, k)
    ideal = dcg_at_k(sorted(relevances, reverse=True), k)
    return 0.0 if ideal == 0 else float(dcg / ideal)


def precision_at_k(relevances, k=10):
    relevances = list(relevances)[:k]
    if len(relevances) == 0:
        return 0.0

    # Jika kandidat kurang dari k, denominator tetap k sesuai definisi Precision@K.
    binary_rels = [1 if rel > 0 else 0 for rel in relevances]
    return float(sum(binary_rels) / k)


def build_qrels_dict(qrels_df: pd.DataFrame):
    qrels_std = standardize_qrels_df(qrels_df)
    return {
        (row.query_id, row.doc_id): int(row.score)
        for row in qrels_std.itertuples(index=False)
    }


def evaluate_ranked_dataframe(ranked_df: pd.DataFrame, qrels_df: pd.DataFrame, k=10):
    """Evaluasi dataframe hasil ranking dengan kolom query_id, doc_id, rank."""
    qrels_dict = build_qrels_dict(qrels_df)

    all_mrr = []
    all_ndcg = []
    all_precision = []

    for qid, group in ranked_df.groupby("query_id", sort=False):
        group = group.sort_values("rank")
        relevances = [
            qrels_dict.get((str(qid), str(doc_id)), 0)
            for doc_id in group["doc_id"].tolist()
        ]

        all_mrr.append(mrr_at_k(relevances, k))
        all_ndcg.append(ndcg_at_k(relevances, k))
        all_precision.append(precision_at_k(relevances, k))

    return {
        f"MRR@{k}": float(np.mean(all_mrr)) if all_mrr else 0.0,
        f"NDCG@{k}": float(np.mean(all_ndcg)) if all_ndcg else 0.0,
        f"Precision@{k}": float(np.mean(all_precision)) if all_precision else 0.0,
        "num_queries": len(all_mrr)
    }

## 3.4 Re-Ranking Candidate Passage dengan BERT Cross-Encoder

Jika variabel `reranker` dari Part 2 sudah ada, cell ini akan memakainya. Jika belum ada, cell ini akan membuat ulang model pre-trained `cross-encoder/ms-marco-MiniLM-L-6-v2`.

In [ ]:
def get_or_create_reranker():
    """Memakai reranker dari Part 2 jika ada, jika tidak membuat model pre-trained."""
    global BERTCrossEncoder

    if "reranker" in globals():
        print("Memakai variabel reranker yang sudah ada dari Part 2.")
        return reranker

    if "BERTCrossEncoder" not in globals():
        try:
            from src.bert_cross_encoder import BERTCrossEncoder
            print("BERTCrossEncoder berhasil di-import ulang.")
        except ImportError:
            raise NameError("BERTCrossEncoder belum tersedia dan gagal di-import. Pastikan folder 'src' ada.")

    print("Variabel reranker belum ada. Membuat BERT Cross-Encoder pre-trained...")
    return BERTCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank_candidates_for_qa(reranker_model, candidate_df: pd.DataFrame, top_k=10, batch_size=8):
    """Re-rank candidate passage untuk setiap query."""
    if candidate_df is None or len(candidate_df) == 0:
        print("Warning: candidate_df kosong. Melewati re-ranking.")
        return pd.DataFrame()

    rows = []

    for qid, group in tqdm(candidate_df.groupby("query_id", sort=False), desc="Re-ranking queries"):
        group = group.reset_index(drop=True)
        query_text = group.loc[0, "query"]
        passages = group["passage"].tolist()
        doc_ids = group["doc_id"].tolist()

        ranked_indices, scores = reranker_model.re_rank(
            query=query_text,
            passages=passages,
            batch_size=batch_size,
            verbose=False
        )

        for rank, idx in enumerate(ranked_indices[:top_k], start=1):
            rows.append({
                "query_id": str(qid),
                "query": query_text,
                "doc_id": str(doc_ids[idx]),
                "passage": passages[idx],
                "rank": rank,
                "reranker_score": float(scores[idx])
            })

    return pd.DataFrame(rows)


# Main Execution
try:
    part3_reranker = get_or_create_reranker()

    if 'part3_candidate_df' in globals() and part3_candidate_df is not None:
        part3_reranked_df = rerank_candidates_for_qa(
            reranker_model=part3_reranker,
            candidate_df=part3_candidate_df,
            top_k=PART3_RERANK_TOP_K,
            batch_size=PART3_BATCH_SIZE
        )

        if not part3_reranked_df.empty:
            part3_ranking_metrics = evaluate_ranked_dataframe(
                ranked_df=part3_reranked_df,
                qrels_df=part3_qrels_df,
                k=10
            )

            print("Ranking metrics:")
            display(pd.DataFrame([part3_ranking_metrics]))
            display(part3_reranked_df.head(10))
    else:
        print("Error: part3_candidate_df tidak tersedia. Silakan jalankan cell di bagian 3.2 terlebih dahulu.")
except Exception as e:
    print(f"Terjadi kesalahan: {e}")

## 3.5 Extractive QA pada Top Passage

Untuk setiap query, QA pipeline dijalankan pada beberapa passage teratas hasil re-ranking. Jawaban dengan `qa_score` tertinggi dipilih sebagai jawaban final.

In [ ]:
def safe_question_answering(qa_pipe, question: str, context: str):
    """Wrapper agar QA pipeline tetap aman jika parameter tertentu tidak didukung versi transformers."""
    try:
        return qa_pipe(
            question=question,
            context=context,
            handle_impossible_answer=True,
            max_answer_len=50
        )
    except TypeError:
        return qa_pipe(
            question=question,
            context=context
        )


def run_extractive_qa_on_reranked(
    qa_pipe,
    reranked_df: pd.DataFrame,
    top_context_k=5,
    min_score=0.05
):
    """
    Menjalankan Extractive QA pada top-k passage hasil re-ranking.
    """
    if reranked_df is None or reranked_df.empty:
        print("Warning: reranked_df kosong atau tidak tersedia. Melewati QA.")
        return pd.DataFrame()

    rows = []

    for qid, group in tqdm(reranked_df.groupby("query_id", sort=False), desc="Running Extractive QA"):
        group = group.sort_values("rank").head(top_context_k)
        question = group.iloc[0]["query"]

        best_answer = None

        for _, row in group.iterrows():
            context = str(row["passage"])
            if context.strip() == "":
                continue

            qa_result = safe_question_answering(
                qa_pipe=qa_pipe,
                question=question,
                context=context
            )

            answer_raw = str(qa_result.get("answer", "")).strip()
            qa_score = float(qa_result.get("score", 0.0))

            candidate_answer = {
                "query_id": str(qid),
                "query": question,
                "answer_raw": answer_raw,
                "qa_score": qa_score,
                "source_doc_id": str(row["doc_id"]),
                "source_rank": int(row["rank"]),
                "source_reranker_score": float(row["reranker_score"]),
                "source_passage": context
            }

            if best_answer is None or qa_score > best_answer["qa_score"]:
                best_answer = candidate_answer

        if best_answer is None:
            best_answer = {
                "query_id": str(qid),
                "query": group.iloc[0]["query"],
                "answer_raw": "",
                "qa_score": 0.0,
                "source_doc_id": None,
                "source_rank": None,
                "source_reranker_score": None,
                "source_passage": ""
            }

        if best_answer["answer_raw"] == "" or best_answer["qa_score"] < min_score:
            best_answer["answer_final"] = "Tidak ditemukan jawaban eksplisit pada top passage."
        else:
            best_answer["answer_final"] = best_answer["answer_raw"]

        rows.append(best_answer)

    cols = [
        "query_id", "query", "answer_final", "answer_raw", "qa_score",
        "source_doc_id", "source_rank", "source_reranker_score", "source_passage"
    ]
    return pd.DataFrame(rows)[cols]


# Main Execution
if 'part3_reranked_df' in globals() and part3_reranked_df is not None:
    part3_qa_results_df = run_extractive_qa_on_reranked(
        qa_pipe=qa_pipeline,
        reranked_df=part3_reranked_df,
        top_context_k=PART3_QA_CONTEXT_TOP_K,
        min_score=QA_MIN_SCORE
    )

    if not part3_qa_results_df.empty:
        pd.set_option("display.max_colwidth", 160)
        display(part3_qa_results_df[[
            "query_id", "query", "answer_final", "qa_score", "source_doc_id", "source_rank"
        ]])
else:
    print("Error: 'part3_reranked_df' tidak ditemukan. Pastikan cell di Part 3.4 sudah dijalankan dengan sukses.")

## 3.6 Evaluasi Sumber Jawaban Extractive QA

Karena dataset IR biasanya menyediakan qrels dokumen, bukan gold answer span, evaluasi utama Part 3 dilakukan dengan dua cara:

1. **Evaluasi ranking**: MRR@10, NDCG@10, Precision@10 dari urutan Cross-Encoder.
2. **Evaluasi sumber jawaban**: apakah passage yang dipilih sebagai sumber jawaban memiliki label relevan di qrels.

Jika dosen menyediakan gold answer, metrik EM/F1 dapat ditambahkan, tetapi pada notebook ini evaluasi dibuat berdasarkan qrels yang sudah tersedia.

In [ ]:
def evaluate_qa_source_document(qa_results_df: pd.DataFrame, qrels_df: pd.DataFrame):
    """Evaluasi apakah dokumen sumber jawaban QA adalah dokumen relevan menurut qrels."""
    qrels_dict = build_qrels_dict(qrels_df)
    eval_rows = []

    for row in qa_results_df.itertuples(index=False):
        qid = str(row.query_id)
        doc_id = str(row.source_doc_id)
        relevance = qrels_dict.get((qid, doc_id), 0)
        source_rank = row.source_rank if pd.notna(row.source_rank) else None

        eval_rows.append({
            "query_id": qid,
            "source_doc_id": doc_id,
            "source_rank": source_rank,
            "source_relevance": int(relevance),
            "source_is_relevant": int(relevance > 0),
            "source_mrr": (1.0 / source_rank) if (source_rank and relevance > 0) else 0.0,
            "qa_score": float(row.qa_score)
        })

    detail_df = pd.DataFrame(eval_rows)

    summary = {
        "QA_Source_Hit_Rate": float(detail_df["source_is_relevant"].mean()) if len(detail_df) else 0.0,
        "QA_Source_MRR": float(detail_df["source_mrr"].mean()) if len(detail_df) else 0.0,
        "Avg_QA_Score": float(detail_df["qa_score"].mean()) if len(detail_df) else 0.0,
        "Avg_Source_Rank": float(detail_df["source_rank"].dropna().mean()) if detail_df["source_rank"].notna().any() else 0.0,
        "num_queries": int(len(detail_df))
    }

    return summary, detail_df


part3_qa_source_metrics, part3_qa_source_detail_df = evaluate_qa_source_document(
    qa_results_df=part3_qa_results_df,
    qrels_df=part3_qrels_df
)

print("QA source document metrics:")
display(pd.DataFrame([part3_qa_source_metrics]))

display(part3_qa_source_detail_df)

## 3.7 Ringkasan Akhir Part 3

Tabel berikut menggabungkan metrik ranking dan metrik sumber jawaban QA.

In [ ]:
part3_summary_df = pd.DataFrame([
    {
        "Dataset": PART3_DATASET,
        "Model": "BERT Cross-Encoder + Extractive QA",
        "MRR@10": part3_ranking_metrics.get("MRR@10", 0.0),
        "NDCG@10": part3_ranking_metrics.get("NDCG@10", 0.0),
        "Precision@10": part3_ranking_metrics.get("Precision@10", 0.0),
        "QA_Source_Hit_Rate": part3_qa_source_metrics.get("QA_Source_Hit_Rate", 0.0),
        "QA_Source_MRR": part3_qa_source_metrics.get("QA_Source_MRR", 0.0),
        "Avg_QA_Score": part3_qa_source_metrics.get("Avg_QA_Score", 0.0),
        "Num Queries": part3_ranking_metrics.get("num_queries", 0)
    }
])

display(part3_summary_df)

## 3.8 Simpan Hasil Part 3

Hasil disimpan ke folder `outputs` supaya bisa dipakai untuk laporan.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

part3_reranked_path = OUTPUT_DIR / "part3_reranked_results.csv"
part3_qa_path = OUTPUT_DIR / "part3_qa_results.csv"
part3_summary_path = OUTPUT_DIR / "part3_summary.csv"

part3_reranked_df.to_csv(part3_reranked_path, index=False)
part3_qa_results_df.to_csv(part3_qa_path, index=False)
part3_summary_df.to_csv(part3_summary_path, index=False)

print("Saved:")
print("-", part3_reranked_path)
print("-", part3_qa_path)
print("-", part3_summary_path)